# **Space X Falcon 9 First Stage Landing Prediction**

## Data Collection — SpaceX REST API

Estimated time needed: **30** minutes

In this lab, we will collect Falcon 9 launch data from the SpaceX public REST API.
SpaceX advertises Falcon 9 rocket launches on its website with a cost of 62 million dollars;
other providers cost upward of 165 million dollars each. Much of the savings is because
SpaceX can reuse the first stage. If we can predict whether the first stage will land,
we can estimate the cost of a launch.

### Objectives

- Make GET requests to the SpaceX API
- Extract and parse launch data from JSON responses
- Build a flat Pandas DataFrame with all relevant features
- Save the dataset for downstream analysis

---
## Import Libraries

In [1]:
import requests
import pandas as pd
import numpy as np
import datetime
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
print('Libraries loaded successfully.')

Libraries loaded successfully.


## Define Helper Functions

Below we define helper functions that take a SpaceX API launch record and
resolve the related IDs (rocket, launchpad, payload, core) into readable features.

Each helper populates **global lists** that will later become DataFrame columns.

### API call flow

```
GET /v4/launches/past
  └─ For each launch:
      ├─ GET /v4/rockets/{id}      → BoosterVersion
      ├─ GET /v4/launchpads/{id}   → LaunchSite, Latitude, Longitude
      ├─ GET /v4/payloads/{id}     → PayloadMass, Orbit
      └─ GET /v4/cores/{id}        → Serial, Block, ReusedCount,
                                     Outcome, Flights, GridFins,
                                     Reused, Legs, LandingPad
```

In [2]:
# Global lists — one per feature column
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

BASE_URL = 'https://api.spacexdata.com/v4'

In [3]:
def getBoosterVersion(data):
    """Resolve rocket ID to rocket name for each launch."""
    for rocket_id in data['rocket']:
        try:
            response = requests.get(f"{BASE_URL}/rockets/{rocket_id}", timeout=10).json()
            BoosterVersion.append(response.get('name', None))
        except Exception:
            BoosterVersion.append(None)


def getLaunchSite(data):
    """Resolve launchpad ID to site name, latitude, and longitude."""
    for pad_id in data['launchpad']:
        try:
            response = requests.get(f"{BASE_URL}/launchpads/{pad_id}", timeout=10).json()
            LaunchSite.append(response.get('name', None))
            Longitude.append(response.get('longitude', None))
            Latitude.append(response.get('latitude', None))
        except Exception:
            LaunchSite.append(None)
            Longitude.append(None)
            Latitude.append(None)


def getPayloadData(data):
    """Resolve payload IDs to mass and orbit."""
    for payload_ids in data['payloads']:
        if payload_ids:
            pid = payload_ids[0] if isinstance(payload_ids, list) else payload_ids
            try:
                response = requests.get(f"{BASE_URL}/payloads/{pid}", timeout=10).json()
                PayloadMass.append(response.get('mass_kg', None))
                Orbit.append(response.get('orbit', None))
            except Exception:
                PayloadMass.append(None)
                Orbit.append(None)
        else:
            PayloadMass.append(None)
            Orbit.append(None)


def getCoreData(data):
    """Resolve core ID to serial, block, reuse count, and landing outcome."""
    for cores in data['cores']:
        core = cores[0] if isinstance(cores, list) and len(cores) > 0 else cores
        if isinstance(core, dict):
            # Resolve core detail
            if core.get('core'):
                try:
                    response = requests.get(f"{BASE_URL}/cores/{core['core']}", timeout=10).json()
                    Block.append(response.get('block', None))
                    ReusedCount.append(response.get('reuse_count', None))
                    Serial.append(response.get('serial', None))
                except Exception:
                    Block.append(None)
                    ReusedCount.append(None)
                    Serial.append(None)
            else:
                Block.append(None)
                ReusedCount.append(None)
                Serial.append(None)

            Outcome.append(str(core.get('landing_success', None)) + ' ' + str(core.get('landing_type', None)))
            Flights.append(core.get('flight', None))
            GridFins.append(core.get('gridfins', None))
            Reused.append(core.get('reused', None))
            Legs.append(core.get('legs', None))
            LandingPad.append(core.get('landpad', None))
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
            Outcome.append(None)
            Flights.append(None)
            GridFins.append(None)
            Reused.append(None)
            Legs.append(None)
            LandingPad.append(None)

print('Helper functions defined.')

Helper functions defined.


## Request Launch Data from SpaceX API

We request all past launches and filter for **Falcon 9** (rocket ID `5e9d0d95eda69973a809d1ec`).

In [4]:
FALCON9_ID = '5e9d0d95eda69973a809d1ec'

response = requests.get(f'{BASE_URL}/launches/past', timeout=30)
print(f'API Status Code: {response.status_code}')

all_launches = response.json()
print(f'Total past launches: {len(all_launches)}')

# Filter Falcon 9 only
falcon9_launches = [l for l in all_launches if l.get('rocket') == FALCON9_ID]
print(f'Falcon 9 launches: {len(falcon9_launches)}')

API Status Code: 200
Total past launches: 187
Falcon 9 launches: 179


## Build an Initial DataFrame from the JSON

Extract the top-level fields (flight number, date, mission name, payloads, cores, etc.) into a Pandas DataFrame, then use the helper functions to resolve related IDs.

In [5]:
# Create a flat list of records with top-level fields
records = []
for launch in falcon9_launches:
    record = {
        'FlightNumber': launch.get('flight_number'),
        'Date': launch.get('date_utc', '')[:10],
        'rocket': launch.get('rocket'),
        'launchpad': launch.get('launchpad'),
        'payloads': launch.get('payloads', []),
        'cores': launch.get('cores', [{}])[0] if launch.get('cores') else {}
    }
    records.append(record)

data = pd.DataFrame(records)
print(f'Initial DataFrame shape: {data.shape}')
data.head()

Initial DataFrame shape: (179, 6)


,FlightNumber,Date,rocket,launchpad,payloads,cores
0,6,2010-06-04,5e9d0d95eda69973a809d1ec,5e9e4501f509094ba4566f84,[5eb0e4b7b6c3bb0006eeb1e7],"{'core': '5e9e289ef359185f2b3b2628', 'flight':..."
1,7,2010-12-08,5e9d0d95eda69973a809d1ec,5e9e4501f509094ba4566f84,"[5eb0e4b9b6c3bb0006eeb1e8, 5eb0e4b9b6c3bb0006e...","{'core': '5e9e289ef35918187c3b2629', 'flight':..."
2,8,2012-05-22,5e9d0d95eda69973a809d1ec,5e9e4501f509094ba4566f84,[5eb0e4bab6c3bb0006eeb1ea],"{'core': '5e9e289ef35918f39c3b262a', 'flight':..."
3,9,2012-10-08,5e9d0d95eda69973a809d1ec,5e9e4501f509094ba4566f84,"[5eb0e4bab6c3bb0006eeb1eb, 5eb0e4bab6c3bb0006e...","{'core': '5e9e289ff3591821a73b262b', 'flight':..."
4,10,2013-03-01,5e9d0d95eda69973a809d1ec,5e9e4501f509094ba4566f84,[5eb0e4bbb6c3bb0006eeb1ed],"{'core': '5e9e289ff3591884e03b262c', 'flight':..."


## Resolve IDs Using Helper Functions

We now call each helper to populate the global lists by making additional API requests.

**Note:** This takes a few minutes due to the number of API calls.

In [6]:
# Reset all global lists
BoosterVersion = []
PayloadMass = []
Orbit = []
LaunchSite = []
Outcome = []
Flights = []
GridFins = []
Reused = []
Legs = []
LandingPad = []
Block = []
ReusedCount = []
Serial = []
Longitude = []
Latitude = []

print('Resolving Booster Versions ...')
getBoosterVersion(data)

print('Resolving Launch Sites ...')
getLaunchSite(data)

print('Resolving Payload Data ...')
getPayloadData(data)

print('Resolving Core Data ...')
getCoreData(data)

print(f'\nAll helpers complete.  Lengths — BoosterVersion: {len(BoosterVersion)}, '
      f'LaunchSite: {len(LaunchSite)}, PayloadMass: {len(PayloadMass)}, '
      f'Outcome: {len(Outcome)}')

Resolving Booster Versions ...


Resolving Launch Sites ...


Resolving Payload Data ...


Resolving Core Data ...



All helpers complete.  Lengths — BoosterVersion: 179, LaunchSite: 179, PayloadMass: 179, Outcome: 179


## Assemble the Final Dataset

Combine the top-level fields with the resolved features into a single DataFrame.

In [7]:
launch_data = {
    'FlightNumber': list(data['FlightNumber']),
    'Date': list(data['Date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass': PayloadMass,
    'Orbit': Orbit,
    'LaunchSite': LaunchSite,
    'Outcome': Outcome,
    'Flights': Flights,
    'GridFins': GridFins,
    'Reused': Reused,
    'Legs': Legs,
    'LandingPad': LandingPad,
    'Block': Block,
    'ReusedCount': ReusedCount,
    'Serial': Serial,
    'Longitude': Longitude,
    'Latitude': Latitude
}

df = pd.DataFrame(launch_data)
print(f'Final dataset shape: {df.shape}')
df.head()

Final dataset shape: (179, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,6,2010-06-04,Falcon 9,NaN,LEO,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0003,-80.577366,28.561857
1,7,2010-12-08,Falcon 9,NaN,LEO,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0004,-80.577366,28.561857
2,8,2012-05-22,Falcon 9,525.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0005,-80.577366,28.561857
3,9,2012-10-08,Falcon 9,400.0,ISS,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0006,-80.577366,28.561857
4,10,2013-03-01,Falcon 9,677.0,ISS,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0007,-80.577366,28.561857


## Filter for Falcon 9 Launches Only

We keep only `Falcon 9` booster version rows (exclude any Falcon 1 rows that may have slipped through).

In [8]:
df = df[df['BoosterVersion'] == 'Falcon 9'].reset_index(drop=True)

# Re-number flight numbers sequentially for Falcon 9
df['FlightNumber'] = list(range(1, len(df) + 1))

print(f'Falcon 9 only — shape: {df.shape}')
df.head()

Falcon 9 only — shape: (179, 17)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,NaN,LEO,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0003,-80.577366,28.561857
1,2,2010-12-08,Falcon 9,NaN,LEO,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0004,-80.577366,28.561857
2,3,2012-05-22,Falcon 9,525.0,LEO,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0005,-80.577366,28.561857
3,4,2012-10-08,Falcon 9,400.0,ISS,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0006,-80.577366,28.561857
4,5,2013-03-01,Falcon 9,677.0,ISS,CCSFS SLC 40,None None,1,False,False,False,None,1,0,B0007,-80.577366,28.561857


## Data Inspection

In [9]:
print('--- Data Types ---')
print(df.dtypes)
print('\n--- Null Counts ---')
print(df.isnull().sum())
print('\n--- Unique Outcome Values ---')
print(df['Outcome'].value_counts())

--- Data Types ---
FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block               int64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64
dtype: object

--- Null Counts ---
FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass       23
Orbit              1
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        31
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

--- Unique Outcome Values ---
Outcome
True ASDS      114
None None       24
True RTLS       23
False ASDS       8
True Ocean       5


## Handle Missing Values

Replace missing `PayloadMass` values with the column mean.

In [10]:
df['PayloadMass'] = df['PayloadMass'].astype(float)
mean_payload = df['PayloadMass'].mean()
df['PayloadMass'].fillna(mean_payload, inplace=True)

print(f'PayloadMass nulls after imputation: {df["PayloadMass"].isnull().sum()}')
print(f'Mean payload mass used: {mean_payload:.2f} kg')

PayloadMass nulls after imputation: 0
Mean payload mass used: 8117.57 kg


## Save to CSV

Export the cleaned dataset for use in downstream notebooks.

In [11]:
os.makedirs('../data', exist_ok=True)
output_path = '../data/dataset_part_1.csv'
df.to_csv(output_path, index=False)
print(f'Saved {len(df)} rows × {len(df.columns)} columns to {output_path}')
print(f'\nColumns: {list(df.columns)}')

Saved 179 rows × 17 columns to ../data/dataset_part_1.csv

Columns: ['FlightNumber', 'Date', 'BoosterVersion', 'PayloadMass', 'Orbit', 'LaunchSite', 'Outcome', 'Flights', 'GridFins', 'Reused', 'Legs', 'LandingPad', 'Block', 'ReusedCount', 'Serial', 'Longitude', 'Latitude']


## Summary

In this notebook we:

1. **Requested** all past SpaceX launches from the v4 REST API
2. **Filtered** for Falcon 9 launches only
3. **Resolved** related entity IDs (rocket, launchpad, payload, core) into readable features
4. **Built** the `Outcome` column as `"{landing_success} {landing_type}"` for downstream classification
5. **Imputed** missing `PayloadMass` values with the column mean
6. **Saved** the final dataset as `data/dataset_part_1.csv`

The resulting columns are:

| Column | Description |
|--------|-------------|
| FlightNumber | Sequential launch number |
| Date | Launch date (UTC) |
| BoosterVersion | Rocket name (Falcon 9) |
| PayloadMass | Payload mass in kg |
| Orbit | Target orbit |
| LaunchSite | Launch pad name |
| Outcome | `"{landing_success} {landing_type}"` |
| Flights | Number of flights for this core |
| GridFins | Whether grid fins were used |
| Reused | Whether core was reused |
| Legs | Whether landing legs were used |
| LandingPad | Landing pad ID |
| Block | Core block number |
| ReusedCount | Number of times core reused |
| Serial | Core serial number |
| Longitude | Launch site longitude |
| Latitude | Launch site latitude |